# LoRA Fine-tune Gemma 4 E2B — Resume-Job Matcher
Runs on Colab A100 (40 GB). Training ~3hr min for 3 epochs over 9k examples.

In [1]:
# Cell 1 — Install
%pip install --quiet unsloth datasets trl

In [2]:
# Cell 2 — Mount Drive and set data path
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/finetune_data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3 — Load model + apply LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-e2b-it",
    max_seq_length=3072,  
    load_in_4bit=False,  # Disabled 4-bit! We have an A100, so we run in blazing-fast native bf16.
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,      
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [4]:
# Cell 4 — Load dataset
import os
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": os.path.join(DATA_DIR, "train.jsonl"),
    "validation": os.path.join(DATA_DIR, "val.jsonl"),
})
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 997
    })
})


In [5]:
# Cell 5 — Verify sequence lengths against budget (sanity check before training)
import numpy as np

sample = ds["train"].select(range(min(500, len(ds["train"]))))
# Gemma4Processor wraps a tokenizer at .tokenizer — use that directly for text-only encoding
tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
lengths = [len(tok.encode(ex["text"])) for ex in sample if ex["text"]]
print(f"median={int(np.median(lengths))}  p95={int(np.percentile(lengths, 95))}  p99={int(np.percentile(lengths, 99))}  max={max(lengths)}")
over = sum(1 for l in lengths if l > 4096)
print(f"{over}/{len(lengths)} sample rows exceed 4096 tokens")

median=1467  p95=1602  p99=1647  max=1766
0/500 sample rows exceed 4096 tokens


In [ ]:
# Cell 6 — Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=4096,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,   # effective batch = 16
        num_train_epochs=3,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_steps=60,               # ~56 warmup steps over 1125 total
        bf16=True, #a100 supports bf16 and is fast
        fp16=False, #T4 only supports fp16
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=250,
        save_steps=250,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 3 | Total steps = 1,689
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


In [ ]:
# Cell 6b — Save training loss chart + artifacts to Drive (run immediately after Cell 6)
import json, os, shutil
import matplotlib.pyplot as plt

state_path = "outputs/trainer_state.json"
with open(state_path) as f:
    state = json.load(f)

train_steps, train_losses = [], []
eval_steps,  eval_losses  = [], []
for entry in state["log_history"]:
    if "loss" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])

print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final eval  loss : {eval_losses[-1]:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_losses, label="train loss")
plt.plot(eval_steps,  eval_losses,  label="eval loss", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss — Gemma 4 E2B Resume LoRA")
plt.legend()
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()

DRIVE_OUT = "/content/drive/MyDrive/finetune_data"
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy("training_loss.png", DRIVE_OUT)
shutil.copy(state_path, DRIVE_OUT)
print("Saved training_loss.png and trainer_state.json to Drive")

In [ ]:
# Cell 7 — Save LoRA adapter
model.save_pretrained("gemma4-e2b-resume-lora")
tokenizer.save_pretrained("gemma4-e2b-resume-lora")
print("Adapter saved to gemma4-e2b-resume-lora/")

In [ ]:
# Cell 8b — Standalone GGUF conversion (fresh session, no retraining needed)
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

%pip install --quiet unsloth

# Unzip the LoRA adapter from Drive
LORA_ZIP = "/content/drive/MyDrive/finetune_data/gemma4-e2b-resume-lora.zip"  # adjust if named differently
shutil.unpack_archive(LORA_ZIP, "gemma4-e2b-resume-lora")

# Load base model + LoRA together — Unsloth handles the merge internally for GGUF export
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="gemma4-e2b-resume-lora",  # points at the unzipped adapter directory
    max_seq_length=4096,
    load_in_4bit=False,
)

# Convert to GGUF
OUT_DIR = "gemma4-e2b-resume-gguf"
model.save_pretrained_gguf(OUT_DIR, tokenizer, quantization_method="q4_k_m")

# Copy to Drive
DRIVE_OUT = "/content/drive/MyDrive/finetune_data"
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in os.listdir(OUT_DIR):
    if f.endswith(".gguf"):
        shutil.copy(os.path.join(OUT_DIR, f), DRIVE_OUT)
        print(f"Copied {f} to Drive")

## Cell 8b — GGUF conversion fallback (run in a fresh Colab session if Cell 8 failed)

Use this if `save_pretrained_gguf` threw a llama.cpp compilation error.  
You only need your LoRA zip on Drive — no retraining required.

1. Start a new Colab notebook with an A100 (or T4) runtime  
2. Run the cell below

## Loading in LM Studio on M1 Mac

### 1. Get the GGUF onto the Mac

Download `gemma4-e2b-resume-gguf.zip` from your Drive and unzip it. You'll get two files:
- `gemma-4-e2b-it.Q4_K_M.gguf` — ~1.2 GB, fastest, good default
- `gemma-4-e2b-it.Q5_K_M.gguf` — ~1.5 GB, slightly sharper outputs

### 2. Sideload into LM Studio

1. Open LM Studio → **My Models** tab → click the models directory path to reveal it in Finder
2. Create `lmstudio-community/gemma-4-e2b-resume/` inside that folder
3. Drop both `.gguf` files in
4. Hit refresh in LM Studio — model appears in the list

If your LM Studio has **Import GGUF** in the UI, use that instead.

### 3. Set the chat template (do not skip this)

In LM Studio, load the model from the **Chat** tab, open the right panel → **Prompt Template**, and pick the **Gemma** preset. If it's not listed, set a custom template:

```
<start_of_turn>user
{prompt}<end_of_turn>
<start_of_turn>model
```

Add `<end_of_turn>` as a **stop string**. Set **Temperature 0.1**, **Top P 0.95**.

### 4. Test prompt

```
Analyze the match between this resume and job posting.
Return a JSON object with: match_score, experience_level_fit, rationale,
matching_strengths, skill_gaps, ats_keywords_missing, resume_improvements,
recommended_activities.

RESUME:
<paste resume>

JOB POSTING:
<paste job posting>
```

Expected output: well-formed JSON with all 8 fields. If you get generic Gemma text instead, the chat template is wrong.

### Memory on M1

Q4_K_M at 2B uses ~1.5 GB resident + ~1 GB KV cache at 4096 ctx. Runs fine on any M1. Expect 30-50 tok/s on base M1, 60-90 on M1 Pro/Max.

### If LM Studio refuses to load

Gemma 4 is recent — update LM Studio first. Verify by loading `unsloth/gemma-4-e2b-it-GGUF` from the LM Studio search; if the base model loads, your fine-tune will too.